In [1]:
# Install dependencies if not already present
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'pandas', 'requests',
                'ipywidgets', '--quiet'], check=False)



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


CompletedProcess(args=['/Users/conwaymc/Documents/workspace-nexus/GeoNexus_CDC_Places_MVP_Vignette/.venv/bin/python', '-m', 'pip', 'install', 'pandas', 'requests', 'ipywidgets', '--quiet'], returncode=0)

In [2]:
import sys, os

# ── Make sure cdc_places_fetch.py is importable from the same folder ──────────
HERE = os.path.dirname(os.path.abspath('__file__'))
if HERE not in sys.path:
    sys.path.insert(0, HERE)

from cdc_places_fetch import (
    discover_datasets, fetch_places_data, fetch_metadata,
    print_metadata, save_metadata_csv, build_where_clause,
    DEFAULT_QUERY
)

import ipywidgets as w
from IPython.display import display, HTML, clear_output
import pandas as pd


In [3]:

# ── Styling ────────────────────────────────────────────────────────────────────
display(HTML('''
<style>
  .places-ui { font-family: "Inter", "Segoe UI", sans-serif; max-width: 860px; }
  .places-header {
      background: linear-gradient(135deg, #1a4a6e 0%, #1d6fa4 100%);
      color: white; padding: 18px 24px; border-radius: 8px;
      margin-bottom: 18px;
  }
  .places-header h2 { margin: 0 0 4px 0; font-size: 1.3rem; font-weight: 700; letter-spacing: .3px; }
  .places-header p  { margin: 0; font-size: 0.82rem; opacity: .85; }
  .section-box {
      border: 1px solid #d0dce8; border-radius: 8px;
      padding: 14px 18px; margin-bottom: 14px; background: #f8fbfe;
  }
  .section-label {
      font-size: 0.72rem; font-weight: 700; color: #1a4a6e;
      text-transform: uppercase; letter-spacing: .8px;
      margin-bottom: 10px; border-bottom: 2px solid #1d6fa4;
      padding-bottom: 4px;
  }
  .places-btn {
      background: #1d6fa4 !important; color: white !important;
      border: none !important; border-radius: 6px !important;
      font-weight: 600 !important; padding: 6px 18px !important;
      cursor: pointer;
  }
  .places-btn:hover { background: #155d8a !important; }
  .fetch-btn {
      background: #1a6e3c !important; color: white !important;
      border: none !important; border-radius: 6px !important;
      font-weight: 700 !important; font-size: 1rem !important;
      padding: 8px 32px !important; cursor: pointer;
  }
  .fetch-btn:hover { background: #145530 !important; }
  .output-box {
      border: 1px solid #c8d8e8; border-radius: 8px;
      padding: 14px 18px; background: #f0f5f9; font-size: 0.85rem;
      min-height: 60px;
  }
  .tag-note { font-size:0.75rem; color:#555; margin-top:4px; }
</style>
'''))

# ── Widget definitions ─────────────────────────────────────────────────────────

# — Step 1: Search —
w_search   = w.Text(placeholder='Leave blank to search all PLACES datasets',
                    layout=w.Layout(width='400px'))
w_btn_search = w.Button(description='Search Datasets',
                         button_style='', layout=w.Layout(width='160px'))
w_btn_search.add_class('places-btn')
w_results_label = w.HTML('<span style="color:#888;font-size:.8rem">Run a search first.</span>')

# — Step 2: Pick dataset —
w_pick = w.Select(options=[], rows=8,
                  layout=w.Layout(width='100%', min_height='160px'))

# — Step 3: Filters —
w_state   = w.Text(placeholder='e.g. NC', description='State:',
                    style={'description_width':'70px'},
                    layout=w.Layout(width='200px'))
w_county  = w.Text(placeholder='e.g. Wake', description='County:',
                    style={'description_width':'70px'},
                    layout=w.Layout(width='240px'))
w_measure = w.Text(placeholder='e.g. COPD, DIABETES',
                    description='Measure:',
                    style={'description_width':'70px'},
                    layout=w.Layout(width='260px'))
w_zipcode = w.Text(placeholder='5-digit ZIP / ZCTA', description='ZIP:',
                    style={'description_width':'70px'},
                    layout=w.Layout(width='220px'))
w_lat     = w.FloatText(value=0.0, description='Lat:',
                         style={'description_width':'40px'},
                         layout=w.Layout(width='180px'))
w_lon     = w.FloatText(value=0.0, description='Lon:',
                         style={'description_width':'40px'},
                         layout=w.Layout(width='180px'))
w_radius  = w.IntText(value=25000, description='Radius (m):',
                       style={'description_width':'80px'},
                       layout=w.Layout(width='180px'))
w_use_geo = w.Checkbox(value=False, description='Use lat/lon radius filter',
                        indent=False)

# — Step 4: Options —
w_limit   = w.IntText(value=0, description='Row limit:',
                       style={'description_width':'80px'},
                       layout=w.Layout(width='160px'))
w_meta    = w.Checkbox(value=True,  description='Fetch column metadata', indent=False)
w_preview = w.Checkbox(value=False, description='Preview only (5 rows)', indent=False)
w_out     = w.Text(value='places_data.csv', description='Save to:',
                    style={'description_width':'60px'},
                    layout=w.Layout(width='300px'))

# — Fetch button & output —
w_btn_fetch = w.Button(description='Fetch Data',
                        layout=w.Layout(width='160px', height='38px'))
w_btn_fetch.add_class('fetch-btn')
w_out_area  = w.Output(layout=w.Layout(width='100%'))

# ── Dataset store (populated by search) ───────────────────────────────────────
_datasets = []

# ── Callbacks ─────────────────────────────────────────────────────────────────

def on_search(btn):
    global _datasets
    with w_out_area:
        clear_output()
        term = w_search.value.strip() or DEFAULT_QUERY
        print(f"Searching for '{term}' on data.cdc.gov ...")
        try:
            _datasets = discover_datasets(category=None, query=term, limit=80)
        except Exception as e:
            print(f"Search failed: {e}")
            return
        if not _datasets:
            w_results_label.value = '<span style="color:#c00;font-size:.8rem">No datasets found. Try a different search term.</span>'
            w_pick.options = []
            return
        options = [(f"{d['name']}  [{d['id']}]  updated {d['updated']}", d['id'])
                   for d in _datasets]
        w_pick.options = options
        w_results_label.value = (
            f'<span style="color:#1a6e3c;font-size:.8rem;font-weight:600">'
            f'✓ {len(_datasets)} datasets found — pick one below.</span>'
        )
        print(f"Found {len(_datasets)} datasets.")

def on_fetch(btn):
    with w_out_area:
        clear_output()

        dataset_id = w_pick.value
        if not dataset_id:
            print("⚠  No dataset selected. Run a search and pick one first.")
            return

        # — build where clause —
        lat = lon = None
        if w_use_geo.value and (w_lat.value != 0.0 or w_lon.value != 0.0):
            lat, lon = w_lat.value, w_lon.value

        where = build_where_clause(
            state      = w_state.value.strip()   or None,
            county     = w_county.value.strip()  or None,
            measure    = w_measure.value.strip() or None,
            extra_where= None,
            zipcode    = w_zipcode.value.strip() or None,
            lat=lat, lon=lon,
            radius_m   = w_radius.value,
        )

        limit = w_limit.value if w_limit.value > 0 else None
        if w_preview.value:
            limit = 5

        print(f"Dataset : {dataset_id}")
        if where:
            print(f"Filter  : {where}")
        if limit:
            print(f"Limit   : {limit} rows")
        print()

        # — metadata —
        if w_meta.value:
            print("Fetching column metadata ...")
            try:
                meta = fetch_metadata(dataset_id)
                print_metadata(meta)
                meta_path = f"meta_{dataset_id}.csv"
                save_metadata_csv(meta, meta_path)
            except Exception as e:
                print(f"Metadata error: {e}")

        # — data —
        print("Fetching data ...")
        try:
            df = fetch_places_data(
                dataset_id=dataset_id,
                where=where,
                limit=limit,
                verbose=True,
            )
        except Exception as e:
            print(f"\nFetch error: {e}")
            return

        print(f"\nRetrieved {len(df):,} rows × {len(df.columns)} columns.\n")

        if not w_preview.value:
            out_path = w_out.value.strip() or 'places_data.csv'
            df.to_csv(out_path, index=False)
            print(f"Saved to {out_path}")

        display(HTML('<hr style="margin:10px 0">'))
        display(df.head(20))


w_btn_search.on_click(on_search)
w_btn_fetch.on_click(on_fetch)

# ── Layout ─────────────────────────────────────────────────────────────────────

def section(label, *children):
    return w.VBox([
        w.HTML(f'<div class="section-label">{label}</div>'),
        *children
    ], layout=w.Layout(
        border='1px solid #d0dce8',
        padding='14px 18px',
        margin='0 0 14px 0',
        border_radius='8px',
    ))

header = w.HTML('''
<div class="places-header">
  <h2>CDC PLACES Data Explorer</h2>
  <p>Browse and fetch health prevalence datasets from data.cdc.gov &nbsp;|&nbsp;
     Powered by the Socrata Open Data API</p>
</div>''')

sec_search = section(
    "1 · Find a Dataset",
    w.HBox([w_search, w_btn_search], layout=w.Layout(gap='10px', align_items='center')),
    w_results_label,
    w.HTML('<div class="tag-note">Tip: try "county", "census tract", "ZCTA", "GIS"</div>'),
    w_pick,
)

sec_filters = section(
    "2 · Filters  (all optional)",
    w.HBox([w_state, w_county, w_measure], layout=w.Layout(gap='14px', flex_wrap='wrap')),
    w.HTML('<div class="tag-note" style="margin:8px 0 4px">ZIP / ZCTA filter — best used with a ZCTA-level dataset</div>'),
    w_zipcode,
    w.HTML('<div class="tag-note" style="margin:8px 0 4px">Lat / Lon radius — requires a GIS-friendly dataset</div>'),
    w_use_geo,
    w.HBox([w_lat, w_lon, w_radius], layout=w.Layout(gap='10px')),
)

sec_options = section(
    "3 · Options",
    w.HBox([w_meta, w_preview], layout=w.Layout(gap='24px')),
    w.HBox([w_limit, w_out], layout=w.Layout(gap='14px', margin='8px 0 0 0')),
    w.HTML('<div class="tag-note">Row limit 0 = fetch all matching rows. Output file saved in the same folder as this notebook.</div>'),
)

sec_run = w.VBox([
    w_btn_fetch,
    w.HTML('<hr style="margin:12px 0; border-color:#cde">'),
    w.HTML('<div class="section-label">Output</div>'),
    w_out_area,
])

ui = w.VBox([header, sec_search, sec_filters, sec_options, sec_run],
            layout=w.Layout(max_width='880px'))
display(ui)
